# Brachistochrona — nejrychlejší skluzavka

## Zadání slovy

> Máme dva body: start vlevo nahoře a cíl 2 m vpravo a 0,6 m níž. Mezi ně se má
> postavit skluzavka. Kulička se pustí bez počátečního impulzu, jede jen tíhou,
> tření se zanedbá.
>
> **Jaký tvar skluzavky dopraví kuličku do cíle za nejkratší čas?**

Přímka je nejkratší, ale ne nejrychlejší. Kdo klesne strmě hned na začátku, má
zbytek cesty rychlost navíc — a zaplatí za to delší dráhou. Optimum je někde
mezi, a spadne dokonce **pod úroveň cíle**, aby na konci zase stoupalo.

## Formulace

Dráha se rozseká na $n$ úseků v pevných $x$-ových uzlech. Proměnné jsou jen
výšky $y_1,\dots,y_{n-1}$ vnitřních uzlů — o „křivce“ není v zápisu ani slovo.
Rychlost plyne ze zachování energie, $v(y)=\sqrt{2g\,(y_0-y)}$, a na každé
úsečce je zrychlení konstantní, takže doba průjezdu úsečky je přesně její délka
dělená **průměrnou** rychlostí.

$$
\begin{aligned}
\text{minimize}_{\mathbf y}\quad & T(\mathbf y)=\sum_{i=0}^{n-1}
  \frac{2\sqrt{(x_{i+1}-x_i)^2+(y_{i+1}-y_i)^2}}{v_i+v_{i+1}}
  && \text{doba sjezdu (s)}\\
\text{where}\quad & v_i=\sqrt{2g\,(y_0-y_i)}
  && \text{zachovani energie}\\
\text{subject to}\quad & y_0=0,\qquad y_n=-0{,}6
  && \text{pevne krajni body (m)}\\
& -2 \le y_i \le -10^{-4},\quad i=1,\dots,n-1
  && \text{box}
\end{aligned}
$$

Horní mez $-10^{-4}$ ošetřuje **singularitu na startu**: v bodě $y=y_0$ je
rychlost nulová a $1/v$ diverguje. Průměrná rychlost na úsečce ale nulová není,
takže se integruje přes ni, ne přes okamžitou rychlost.

Účelová funkce není konvexní ani hladce zapsatelná pro CVXPY, takže se řeší
`scipy.optimize.minimize` metodou L-BFGS-B jako **blackbox**: solver o fyzice
nic neví, jen si funkci opakovaně vyhodnotí a gradient si udělá numericky.
Startuje se z přímky.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| pevné uzly $x_0,\dots,x_n$ | `x = np.linspace(0.0, X_CIL, POCET_UZLU)` |
| proměnné $y_1,\dots,y_{n-1}$ | `vysledek.x` (vnitřní uzly) |
| doplnění pevných konců | `cela_draha(y_vnitrni)` |
| $T(\mathbf y)$ | `doba_sjezdu(y)` |
| box $-2 \le y_i \le -10^{-4}$ | `bounds=[(-2.0, -1e-4)] * (POCET_UZLU - 2)` |
| start z přímky | `y_primka[1:-1]` |

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_brachistochrona.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_brachistochrona.py) v repozitáři předmětu.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq, minimize

## Model a řešení

In [ ]:
G = 9.81  # tíhové zrychlení [m/s2]
POCET_UZLU = 51  # @param {type:"slider", min:11, max:101, step:10}
X_CIL = 2.0  # @param {type:"slider", min:0.5, max:4.0, step:0.1}
Y_CIL = -0.6  # @param {type:"slider", min:-1.5, max:-0.1, step:0.1}

x = np.linspace(0.0, X_CIL, POCET_UZLU)  # x-ové uzly jsou pevné


def doba_sjezdu(y):
    # v = sqrt(2 g (y0 - y)); na úsečce je zrychlení konstantní, tak doba = délka / prům. rychlost
    v = np.sqrt(2.0 * G * np.maximum(y[0] - y, 0.0) + 1e-16)
    return float(np.sum(2.0 * np.hypot(np.diff(x), np.diff(y)) / (v[:-1] + v[1:])))


def cela_draha(y_vnitrni):
    return np.concatenate(([0.0], y_vnitrni, [Y_CIL]))  # krajní body jsou pevné


y_primka = np.linspace(0.0, Y_CIL, POCET_UZLU)
vysledek = minimize(lambda y: doba_sjezdu(cela_draha(y)), y_primka[1:-1],
                    method="L-BFGS-B", bounds=[(-2.0, -1e-4)] * (POCET_UZLU - 2))
y_opt = cela_draha(vysledek.x)
cas_primka, cas_opt = doba_sjezdu(y_primka), doba_sjezdu(y_opt)

print(f"proměnných: {POCET_UZLU - 2}")
print(f"po přímce: {cas_primka:.4f} s")
print(f"optimum:   {cas_opt:.4f} s  (o {100 * (1 - cas_opt / cas_primka):.0f} % rychleji)")

## Kontrola, která umí selhat

Úloha má od Johanna Bernoulliho známé přesné řešení: je to **cykloida**. Podíl
jejích dvou parametrických rovnic vyřadí poloměr $R$ a zbude jediná rovnice pro
koncový úhel, kterou dopočítá `brentq`. Diskretizace na konečně mnoha uzlech
musí vyjít o kousek **hůř** než přesná křivka, ale ne o víc než procento —
kdyby vyšla lépe, počítá se něco jiného.

In [ ]:
theta = brentq(lambda t: (t - np.sin(t)) / (1.0 - np.cos(t)) - X_CIL / abs(Y_CIL),
               1e-3, 2.0 * np.pi - 1e-3)
R = abs(Y_CIL) / (1.0 - np.cos(theta))
cas_cykloida = float(np.sqrt(R / G) * theta)  # přesná doba sjezdu po cykloidě
odchylka = 100.0 * (cas_opt / cas_cykloida - 1.0)

print(f"analytická cykloida: {cas_cykloida:.4f} s, odchylka optima {odchylka:+.2f} %")
assert 0.0 < odchylka < 1.0, "optimum se neshoduje se známým řešením"

## Obrázek

In [ ]:
t = np.linspace(0.0, theta, 400)
plt.plot(x, y_primka, label=f"přímka — {cas_primka:.3f} s")
plt.plot(x, y_opt, label=f"optimum — {cas_opt:.3f} s")
plt.plot(R * (t - np.sin(t)), -R * (1.0 - np.cos(t)),
         label=f"analytická cykloida — {cas_cykloida:.3f} s")
plt.xlabel("vodorovná vzdálenost [m]")
plt.ylabel("výška [m]")
plt.title("Nejrychlejší skluzavka mezi dvěma body")
plt.legend()
plt.show()

## Na co se zeptat kódu

1. Zdvojnásobte `POCET_UZLU` — jak se změní odchylka od cykloidy?
2. Posuňte cíl níž než dál (`Y_CIL = -1.5`, `X_CIL = 0.5`). Klesne optimum pořád
   pod úroveň cíle?
3. Napište v `doba_sjezdu` rychlost bez odmocniny. Solver doběhne a vypíše číslo
   — chytí to kontrola?